# Exploration du dataset complet (2014 → 2026)

Données : `data/processed/trafic_3_arcs_complet.csv` (voir `data/processed/README.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("data/processed/trafic_3_arcs_complet.csv", low_memory=False)
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True)
df["datetime"] = df["datetime_utc"].dt.tz_convert("Europe/Paris")

AXES = ["Champs-Elysees", "Convention", "Saints-Peres"]
COULEURS = {"Champs-Elysees": "#2a78d6", "Convention": "#eb6834", "Saints-Peres": "#1baf7a"}
print(df.shape)
df.head()

## Taux de valeurs manquantes par jour

Choisir la variable (`"debit"` ou `"occupation"`). Trait fin = taux journalier, trait épais = moyenne glissante 30 jours.
La ligne pointillée marque le passage de `old_data` aux données récentes du challenge (01/10/2025).

In [ ]:
VARIABLE = "debit"

na_jour = (
    df.assign(na=df[VARIABLE].isna())
    .groupby(["axe", df["datetime"].dt.date])["na"].mean()
    .unstack("axe")
)
na_jour.index = pd.to_datetime(na_jour.index)
debut_recent = df.loc[df["source"] == "recent", "datetime"].min().tz_localize(None)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, sharey=True)
for ax, axe in zip(axes, AXES):
    ax.fill_between(na_jour.index, na_jour[axe], color=COULEURS[axe], alpha=0.25, linewidth=0)
    ax.plot(na_jour.index, na_jour[axe].rolling(30, min_periods=1).mean(), color=COULEURS[axe], linewidth=1.8)
    ax.axvline(debut_recent, color="grey", linestyle="--", linewidth=1)
    ax.set_title(axe, loc="left", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
    ax.set_ylim(0, 1.05)
axes[1].set_ylabel(f"Part des heures sans {VARIABLE}")
axes[-1].xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
axes[-1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))
fig.suptitle(f"Taux de valeurs manquantes par jour ({VARIABLE})", x=0.01, ha="left", fontsize=13)
plt.tight_layout()
plt.show()